# move_jb2 관절 스케일 및 blending 반전 실험
GUI 없이 rbpodo 기본 API를 직접 사용한다.
참조: [move_stop.py](../3rdparty/rbpodo/examples/move_stop.py), [move_j.py](../3rdparty/rbpodo/examples/move_j.py), [data_channel.py](../3rdparty/rbpodo/examples/data_channel.py).

시작→반전점→끝을 합계 100구간으로 분할하며 반전점을 정확히 포함한다.
- `reversal`: 모든 이동 관절이 함께 왕복한다. 관절 속도와 전체 속도가 0을 지나야 하는 대조 모션.
- `corner`: 첫 이동 관절은 반전하고 다른 관절은 계속 전진한다. 관절 속도 0과 전체 정지를 구분하고 코너 절삭을 관찰한다. 단일 관절에서는 reversal과 같다.

같은 웨이포인트에 blending=0/50/100을 각각 적용한다. 기본 4스케일×2경로×3값=24회이며 CASES/PATH_TYPES/BLEND_VALUES로 줄일 수 있다.
기본 simulation, EXECUTE=False. 연결 셀에서 시작·반전·종료 자세를 확인한다.


In [1]:
import time, json, csv
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import rbpodo as rb


In [8]:
ROBOT_IP = "10.0.2.7"
MODE = "simulation"             # "simulation" 또는 "real"
EXECUTE = True
REAL_ACK = False
BASE = None                     # None: 연결 후 현재 자세. 또는 np.array([J1,...,J6]), deg
AMPLITUDE = 30.0                 # 최대 이동량 deg
SPEED = 10.0                     # jb2 speed, deg/s
ACCEL = 20.0                     # jb2 acceleration, deg/s²
SPEED_BAR = 1.0                  # 기본 예제의 0.5 대신 1.0: override 영향 분리
N_SEGMENTS = 100                # 왕복/코너 전체 등록 점 개수, 기본 50+50
RECORD_HZ = 100.0
VELOCITY_WINDOW = 0.1            # 차분 창, s
TIMEOUT = 60.0
CASES = {
    "single":  np.array([1, 0,   0, 0, 0, 0]),
    "equal":   np.array([1, 1,   0, 0, 0, 0]),
    "unequal": np.array([1, 0.5, 0, 0, 0, 0]),
    "swapped": np.array([0.5, 1, 0, 0, 0, 0]),
}
# 예: J3/J5 비교는 np.array([0,0,1,0,0.5,0]) 추가
assert MODE in ("simulation", "real")
assert all(np.isfinite(x) and x > 0 for x in (AMPLITUDE,SPEED,ACCEL,RECORD_HZ,VELOCITY_WINDOW,TIMEOUT))
assert 0 < SPEED_BAR <= 1
for scale in CASES.values():
    assert scale.shape == (6,) and np.isfinite(scale).all() and np.any(scale)

assert isinstance(N_SEGMENTS, int) and not isinstance(N_SEGMENTS, bool) and N_SEGMENTS >= 1

BLEND_VALUES = [0.0, 50.0, 100.0]
PATH_TYPES = ["reversal", "corner"]
# reversal: 모든 이동 관절 왕복, corner: 가장 먼저 움직이는 관절만 반전, 나머지는 계속 전진
assert N_SEGMENTS >= 2
assert BLEND_VALUES and all(np.isfinite(b) and 0 <= b <= 100 for b in BLEND_VALUES)
assert all(p in ("reversal", "corner") for p in PATH_TYPES)

def make_motion(base, scale, path_type, n_segments):
    delta = AMPLITUDE * np.asarray(scale)
    peak = base + delta
    if path_type == "reversal":
        end = base.copy()
    else:
        end = base + 2*delta
        first = np.flatnonzero(delta)[0]
        end[first] = base[first]
    count = n_segments//2
    points = np.vstack([np.linspace(base, peak, count+1)[1:],
                        np.linspace(peak, end, n_segments-count+1)[1:]])
    reversing = np.flatnonzero((peak-base)*(end-peak) < 0)
    return points, peak, end, reversing


## 예상 속도 (가설)
관절들이 동일 진행률로 동기화된다고 가정한다. `max_joint`: 최대 이동 관절 속도가 SPEED.
`joint_L2`: 관절 속도 벡터 크기가 SPEED. 실제 제어기가 어느 가설을 따르는지는 측정으로 확인한다.
속도 바가 1보다 작으면 가감속에도 영향을 줄 수 있으므로 단순 배율로 단정하지 않는다.


In [9]:
for name, scale in CASES.items():
    delta = AMPLITUDE * scale
    max_joint = SPEED * np.abs(delta) / np.max(np.abs(delta))
    joint_l2 = SPEED * np.abs(delta) / np.linalg.norm(delta)
    print(f"{name:8s} Δq={delta} | max_joint={max_joint.round(3)} | L2={joint_l2.round(3)}")


single   Δq=[30.  0.  0.  0.  0.  0.] | max_joint=[10.  0.  0.  0.  0.  0.] | L2=[10.  0.  0.  0.  0.  0.]
equal    Δq=[30. 30.  0.  0.  0.  0.] | max_joint=[10. 10.  0.  0.  0.  0.] | L2=[7.071 7.071 0.    0.    0.    0.   ]
unequal  Δq=[30. 15.  0.  0.  0.  0.] | max_joint=[10.  5.  0.  0.  0.  0.] | L2=[8.944 4.472 0.    0.    0.    0.   ]
swapped  Δq=[15. 30.  0.  0.  0.  0.] | max_joint=[ 5. 10.  0.  0.  0.  0.] | L2=[4.472 8.944 0.    0.    0.    0.   ]


## 연결 및 시작 자세 읽기 (모션 없음)
`BASE=None`이면 현재 모드의 현재 자세를 복사한다. 출력된 모든 목표 자세를 확인한다.
이 노트북은 충돌/관절 가동 범위를 자동 검증하지 않는다.


In [10]:
robot = rb.Cobot(ROBOT_IP)
rc = rb.ResponseCollector()
data_channel = rb.CobotData(ROBOT_IP)

def snapshot():
    packet = data_channel.request_data(1.0)
    if packet is None:
        raise TimeoutError("reqdata 응답 없음")
    s = packet.sdata
    return dict(controller_time=float(s.time), state=int(s.robot_state),
                sim=int(s.real_vs_simulation_mode), speed_bar=float(s.default_speed),
                ref=np.array(s.jnt_ref, dtype=float, copy=True),
                ang=np.array(s.jnt_ang, dtype=float, copy=True))

initial = snapshot()
assert initial['state'] == 1, "로봇 정지 상태에서 시작"
if BASE is None:
    BASE = initial['ref' if initial['sim'] == 1 else 'ang'].copy()
BASE = np.asarray(BASE, dtype=float)
assert BASE.shape == (6,) and np.isfinite(BASE).all()
print("현재 모드:", initial['sim'], "현재 speed bar:", initial['speed_bar'], "BASE:", BASE)
for name, scale in CASES.items():
    for path_type in PATH_TYPES:
        points, peak, end, reversing = make_motion(BASE, scale, path_type, N_SEGMENTS)
        print(name, path_type, "반전 자세:", peak, "끝:", end,
              "등록:", len(points), "반전 관절:", reversing+1)
print("총 실행 수:", len(CASES)*len(PATH_TYPES)*len(BLEND_VALUES))


현재 모드: 1 현재 speed bar: 1.0 BASE: [ 2.99997997e+01  6.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04]
single reversal 반전 자세: [ 5.99997997e+01  6.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04] 끝: [ 2.99997997e+01  6.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04] 등록: 100 반전 관절: [1]
single corner 반전 자세: [ 5.99997997e+01  6.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04] 끝: [ 2.99997997e+01  6.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04] 등록: 100 반전 관절: [1]
equal reversal 반전 자세: [ 5.99997997e+01  9.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04] 끝: [ 2.99997997e+01  6.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04] 등록: 100 반전 관절: [1 2]
equal corner 반전 자세: [ 5.99997997e+01  9.00007019e+01  6.30897982e-03 -1.28354998e-02
  1.30530996e-02 -4.35102003e-04] 끝: [ 2.99997997e+01  

In [11]:
def checked(result):
    rc.error().throw_if_not_empty()
    if not result.is_success():
        raise RuntimeError(f"rbpodo command failed: {result}")


def record_until_done(target, rows, require_motion=False):
    # rbpodo CobotData로 직접 수집. 시작 자세 이동 기록은 분석과 분리한다.
    expected_sim = 1 if MODE == "simulation" else 0
    begin = time.perf_counter()
    stable_since = None
    moved = False
    first_q = None
    while time.perf_counter() - begin < TIMEOUT:
        tick = time.perf_counter()
        s = snapshot()
        q = s['ref'] if expected_sim else s['ang']
        rows.append([tick-begin, s['controller_time'], s['state'], s['sim'], s['speed_bar'], *s['ref'], *s['ang']])
        if s['sim'] != expected_sim:
            raise RuntimeError("실행 중 mode 변경")
        if abs(s['speed_bar'] - SPEED_BAR) > .01:
            raise RuntimeError("실행 중 speed bar 불일치")
        if first_q is None: first_q = q.copy()
        moved = moved or s['state'] != 1 or np.max(np.abs(q-first_q)) > .1
        reached = (moved or not require_motion) and s['state'] == 1 and np.max(np.abs(q-target)) < .05
        stable_since = (stable_since if stable_since is not None else tick) if reached else None
        if stable_since is not None and tick-stable_since >= .3:
            return
        time.sleep(max(0, 1/RECORD_HZ - (time.perf_counter()-tick)))
    raise TimeoutError("모션 완료/목표 도달 대기 시간 초과")


def save_trace(path, rows):
    header = ['host_time','controller_time','state','sim','speed_bar'] + [f'ref{j+1}' for j in range(6)] + [f'ang{j+1}' for j in range(6)]
    with path.open('w', newline='') as f:
        writer = csv.writer(f); writer.writerow(header); writer.writerows(rows)


## 실행
설정 셀의 `EXECUTE=True`로 변경 후 설정 셀을 다시 실행하고, **연결·시작 자세 셀도 다시 실행**한다.
실기는 `MODE="real"`, `REAL_ACK=True`가 필요하다. 각 케이스 전 `move_j`로 BASE에 복귀한다.
오류/커널 interrupt 시 task_stop을 시도하고 부분 기록도 저장한다. 커널 강제 종료 시에는 이 처리가 실행되지 않을 수 있다.


In [12]:
if not EXECUTE:
    print("모션 실행 생략: EXECUTE=False")
else:
    if MODE == "real" and not REAL_ACK:
        raise RuntimeError("실기 실행은 REAL_ACK=True 필요")
    assert snapshot()['state'] == 1, "로봇이 움직이는 중"
    OUT = Path('jb2_scale_results') / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    OUT.mkdir(parents=True)
    config = dict(ip=ROBOT_IP, mode=MODE, base=BASE.tolist(), amplitude=AMPLITUDE,
                  speed=SPEED, accel=ACCEL, speed_bar=SPEED_BAR, record_hz=RECORD_HZ,
                  velocity_window=VELOCITY_WINDOW, blend_values=BLEND_VALUES, path_types=PATH_TYPES, n_segments=N_SEGMENTS,
                  cases={k:v.tolist() for k,v in CASES.items()})
    (OUT/'config.json').write_text(json.dumps(config,indent=2))
    traces = {}
    try:
        checked(robot.set_operation_mode(rc, rb.OperationMode.Simulation if MODE == 'simulation' else rb.OperationMode.Real, timeout=3.0))
        checked(robot.set_speed_bar(rc, SPEED_BAR, timeout=3.0))
        time.sleep(.3)
        manifest = []
        for name, scale in CASES.items():
            for path_type in PATH_TYPES:
                points, peak, target, reversing = make_motion(BASE, scale, path_type, N_SEGMENTS)
                for blend_index, blend in enumerate(BLEND_VALUES):
                    key = f"{name}_{path_type}_b{blend_index}_{blend:g}"
                    checked(robot.move_j(rc, BASE, SPEED, ACCEL, timeout=3.0))
                    record_until_done(BASE, [])
                    checked(robot.move_jb2_clear(rc, timeout=3.0))
                    np.savetxt(OUT/f'{key}_waypoints.csv', points, delimiter=',',
                               header='J1_deg,J2_deg,J3_deg,J4_deg,J5_deg,J6_deg', comments='')
                    for point in points:
                        checked(robot.move_jb2_add(rc, point, SPEED, ACCEL, blend, timeout=3.0))
                    entry = dict(key=key, case=name, path_type=path_type, blend=blend,
                                 peak=peak.tolist(), target=target.tolist(),
                                 reversing=reversing.tolist(), completed=False)
                    manifest.append(entry)
                    (OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
                    rows = []
                    try:
                        checked(robot.move_jb2_run(rc, timeout=3.0))
                        record_until_done(target, rows, require_motion=True)
                        entry['completed'] = True
                    finally:
                        save_trace(OUT/f'{key}.csv', rows)
                        (OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
                    print(key, f"jb2_add {len(points)}개, samples={len(rows)}")
    except BaseException:
        robot.task_stop(rc, timeout=3.0)
        raise
    print('저장 경로:', OUT.resolve())


single_reversal_b0_0 jb2_add 100개, samples=4724
single_reversal_b1_50 jb2_add 100개, samples=3647
single_reversal_b2_100 jb2_add 100개, samples=2566
single_corner_b0_0 jb2_add 100개, samples=4727
single_corner_b1_50 jb2_add 100개, samples=3647
single_corner_b2_100 jb2_add 100개, samples=2555
equal_reversal_b0_0 jb2_add 100개, samples=4725
equal_reversal_b1_50 jb2_add 100개, samples=3646
equal_reversal_b2_100 jb2_add 100개, samples=2566
equal_corner_b0_0 jb2_add 100개, samples=4724
equal_corner_b1_50 jb2_add 100개, samples=3646
equal_corner_b2_100 jb2_add 100개, samples=2566


KeyboardInterrupt: 

## 기록 재분석 및 그래프
로봇 연결 없이도 저장 경로를 지정해 아래 셀만 분석 가능하다(import 셀은 필요).
실기는 jnt_ang, simulation은 jnt_ref를 사용하며 둘 다 원본 CSV에 남긴다.
속도는 host 수신 시각에 대한 0.1초 창 차분. 원시 차분 최대값도 병기한다.
95% peak 유지 시간이 짧다면 정상속도 도달을 단정하지 말고 그래프와 이동량을 확인한다.

미완료 실행은 비교에서 제외하며, 반전 미도달량은 DTW 없이 극값끼리 직접 비교한다.


In [7]:
# OUT = Path("jb2_scale_results/저장된_실험_폴더")
if 'OUT' not in globals():
    print('분석할 OUT 경로를 지정하세요.')
else:
    config = json.loads((OUT/'config.json').read_text())
    manifest = json.loads((OUT/'manifest.json').read_text())
    summary, events, grouped = [], [], {}
    for entry in manifest:
        key = entry['key']
        path = OUT/f'{key}.csv'
        if not entry['completed'] or not path.exists():
            print(key, '미완료: 비교 제외'); continue
        a = np.genfromtxt(path, delimiter=',', skip_header=1, ndmin=2)
        if len(a) < 5:
            print(key, '샘플 부족'); continue
        t = a[:,0]
        q = a[:,5:11] if config['mode'] == 'simulation' else a[:,11:17]
        if not np.all(np.diff(t)>0): raise ValueError('수신 시각이 증가하지 않음')
        v = np.zeros_like(q)
        for i, ti in enumerate(t):
            lo = np.searchsorted(t, ti-config['velocity_window']/2)
            hi = min(len(t)-1, np.searchsorted(t, ti+config['velocity_window']/2, side='right')-1)
            if hi>lo: v[i] = (q[hi]-q[lo])/(t[hi]-t[lo])
        raw = np.gradient(q,t,axis=0)
        linf = np.abs(v).max(axis=1); l2 = np.linalg.norm(v,axis=1)
        near = linf >= .95*linf.max()
        row = dict(key=key, case=entry['case'], path=entry['path_type'], blend=entry['blend'],
                   linf_peak=float(linf.max()), l2_peak=float(l2.max()),
                   near_peak_s=float(np.sum(np.diff(t)*(near[:-1]&near[1:]))))
        row.update({f'J{j+1}_peak':float(x) for j,x in enumerate(np.abs(v).max(axis=0))})
        row.update({f'J{j+1}_raw_peak':float(x) for j,x in enumerate(np.abs(raw).max(axis=0))})
        summary.append(row)
        # 이 경로에는 각 대상 관절마다 의도한 반전이 한 번만 있다.
        # 최대/최소 위치를 직접 비교하므로 DTW가 반전 미도달량을 숨기지 않는다.
        for j in entry['reversing']:
            direction = np.sign(entry['peak'][j]-config['base'][j])
            i = int(np.argmax(direction*q[:,j]))
            observed = 0 < i < len(t)-1 and direction*(q[i,j]-q[0,j]) > .05 and direction*(q[i,j]-q[-1,j]) > .05
            events.append(dict(key=key, joint=j+1, blend=entry['blend'], observed=bool(observed),
                target_deg=entry['peak'][j], measured_extreme_deg=float(q[i,j]),
                deficit_deg=float(direction*(entry['peak'][j]-q[i,j])) if observed else None,
                time_s=float(t[i]), signed_velocity_deg_s=float(v[i,j]),
                l2_speed_at_extreme=float(l2[i])))
        grouped.setdefault((entry['case'],entry['path_type']), []).append((entry,t,q,v))
    for (case,path_type), records in grouped.items():
        # 반전 관절 및 계속 이동하는 관절 모두 보여 준다.
        active = np.flatnonzero(config['cases'][case])
        fig, axes = plt.subplots(len(active),2,figsize=(13,3.5*len(active)),squeeze=False)
        for entry,t,q,v in records:
            for r,j in enumerate(active):
                label=f"blend={entry['blend']:g}"
                axes[r,0].plot(t,q[:,j],label=label)
                axes[r,1].plot(t,v[:,j],label=label)
                if j in entry['reversing']:
                    direction=np.sign(entry['peak'][j]-config['base'][j])
                    i=int(np.argmax(direction*q[:,j]))
                    axes[r,0].plot(t[i],q[i,j],'x')
        for r,j in enumerate(active):
            axes[r,0].axhline(records[0][0]['peak'][j],color='gray',ls=':',label='Corner target')
            axes[r,1].axhline(0,color='gray',ls=':')
            axes[r,0].set_ylabel(f'J{j+1} [deg]');axes[r,1].set_ylabel(f'J{j+1} signed speed [deg/s]')
            for ax in axes[r]: ax.set_xlabel('Time after run ACK [s]');ax.grid(alpha=.3);ax.legend()
        fig.suptitle(f'{case} / {path_type}');fig.tight_layout()
        fig.savefig(OUT/f'{case}_{path_type}_blend.png',dpi=140);plt.show()
    for filename, rows in [('summary.csv',summary),('reversal_errors.csv',events)]:
        if rows:
            with (OUT/filename).open('w',newline='') as f:
                writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    for event in events: print(event)


분석할 OUT 경로를 지정하세요.


## 결과 해석
`*_blend.png`에서 blending별 관절 위치와 **부호 있는 속도**를 겹쳐 본다.
`reversal_errors.csv`는 각 반전 관절의 목표 극값과 측정 극값을 직접 비교한다. 양수 deficit은 미도달, 음수는 초과이며, 반전이 관측되지 않으면 observed=False 및 deficit 공란이다.
샘플링/차분 때문에 표시된 극값의 속도가 정확히 0은 아닐 수 있다. 이는 0속도 유지 시간을 명령한 실험이 아니라 **방향 반전으로 0을 통과하는 실험**이다.

완전 왕복은 blending을 높여도 정지가 필요해 차이가 작을 수 있다. corner의 다관절 모션에서는 반전 관절 속도가 0이어도 다른 관절이 계속 움직일 수 있고, 코너를 잘라 목표 극값에 미달하는지 확인할 수 있다.
100분할의 짧은 구간과 blending=0은 최고속도 도달을 제한할 수 있다. 최대속도가 반드시 증가한다고 단정하지 않고 반전 미도달량·부호 속도·전체 속도를 함께 비교한다.
단일 구간 속도 가설 비교가 필요하면 이전 직선 실험을 사용한다. 이 노트북은 반전 보존을 위해 최소 2구간이 필요하다.
